# Step 7 — BAGEL batch-shape versus mixed-age diagnostic

This is one unattended **correctness diagnostic**, not a performance benchmark. It compares the same two prompt/seed requests under:

1. serial B=1 with an eight-second stagger,
2. homogeneous B=2 with zero stagger, and
3. heterogeneous B=2 with the established eight-second stagger.

All arms use `drop_image`, eager execution, two warm-ups, and two measured requests. Every post-step latent from step 0 through 48 is saved. The analyzer accepts the homogeneous arm only if the trace proves no mixed-age measured batch and at least 32 same-step two-request batches; early singleton steps are recorded rather than misclassified. Timing is diagnostic only because latent capture synchronizes CUDA.

Use an **A100 High-RAM** runtime and run the cells from top to bottom. The long cell emits a heartbeat at least every 20 seconds, owns and cleans up each server, preserves partial evidence on failure, creates an integrity-tested local ZIP, and only then attempts a checksum-verified Drive backup. Three isolated server launches make this longer than Step 6; allow roughly **50–75 minutes after installation/checkpoint setup**, with Colab variability.

## 1. Mount Drive and create local-only active paths

Drive is used only for the final verified ZIP. Active logs, traces, latent files, images, and the checkpoint cache remain under `/content`.

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile

drive.mount('/content/drive')
RUN_ID = 'batch_shape_diagnostic_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
FORK_URL = 'https://github.com/yegurla-sumanth/mstar.git'
RUN_COMMIT = '4f13cda5494a8b35b325ce563c39111b7d28c605'
REPO = Path('/content') / f'mstar_{RUN_ID}'
LOCAL_ROOT = Path('/content/bagel_runs')
RESULT_DIR = LOCAL_ROOT / RUN_ID
DIAGNOSTIC_DIR = RESULT_DIR / 'batch_shape_vs_mixed_age'
CACHE_DIR = Path('/content/hf_cache_bagel')
UV_CACHE_DIR = Path('/content/uv_cache')
DRIVE_BACKUP_DIR = Path('/content/drive/MyDrive/CMU_11868_BAGEL')
for path in (LOCAL_ROOT, RESULT_DIR, CACHE_DIR, UV_CACHE_DIR):
    path.mkdir(parents=True, exist_ok=path != RESULT_DIR)
os.environ['UV_CACHE_DIR'] = str(UV_CACHE_DIR)
print({'run_id': RUN_ID, 'active_results': str(RESULT_DIR), 'drive_backup': str(DRIVE_BACKUP_DIR)})

## 2. Clone the pinned diagnostic implementation and verify hardware

This stops before dependency or checkpoint downloads if the runtime lacks an 80 GB-class GPU, sufficient host RAM, or sufficient local disk.

In [ ]:
subprocess.run(['git', 'clone', '--no-checkout', FORK_URL, str(REPO)], check=True)
subprocess.run(['git', 'checkout', '--detach', RUN_COMMIT], cwd=REPO, check=True)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
assert head == RUN_COMMIT, (head, RUN_COMMIT)
subprocess.run([
    sys.executable, str(REPO / 'experiments/bagel/scripts/environment_probe.py'),
    '--output', str(RESULT_DIR / 'preinstall_environment.json'), '--disk-path', '/content',
    '--require-gpu-memory-gib', '75', '--require-host-memory-gib', '50',
    '--require-disk-free-gib', '55',
], check=True)
source = {'fork_url': FORK_URL, 'requested_commit': RUN_COMMIT, 'resolved_commit': head,
          'git_status': subprocess.check_output(['git', 'status', '--porcelain=v1'], cwd=REPO, text=True)}
(RESULT_DIR / 'source.json').write_text(json.dumps(source, indent=2) + '\n')
print('Hardware preflight passed at', head)

## 3. Create or resume the pinned virtual environment

This is re-runnable within the same Colab VM. The preflight includes the new trace-contract tests and the real PyTorch all-step latent test before any server starts.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv==0.10.0'], check=True)
UV = subprocess.check_output([sys.executable, '-c', 'import shutil; print(shutil.which("uv"))'], text=True).strip()
VENV = REPO / '.venv-bagel'
VENV_PY = VENV / 'bin/python'
if not VENV_PY.exists():
    subprocess.run([UV, 'venv', '--python', '3.12', '--seed', str(VENV)], check=True)
constraints = REPO / 'experiments/bagel/constraints-colab-cu128.txt'
subprocess.run([UV, 'pip', 'install', '--python', str(VENV_PY), '--torch-backend=cu128',
                '--constraint', str(constraints), '-e', f'{REPO}[bagel]', 'matplotlib==3.10.7'], check=True)
subprocess.run([UV, 'pip', 'check', '--python', str(VENV_PY)], check=True)
env = os.environ.copy()
env['MPLBACKEND'] = 'Agg'
env['PATH'] = os.pathsep.join((str(VENV / 'bin'), env['PATH']))
tests = ('test_bagel_image_batch_packing.py', 'test_execution_trace.py',
         'test_bagel_experiment_runtime.py', 'test_bagel_pruned_batch_gate.py',
         'test_bagel_batch_shape_diagnostic.py')
for test in tests:
    subprocess.run([str(VENV_PY), str(REPO / 'test/modular' / test), '-v'], cwd=REPO, env=env, check=True)
gpu_check = subprocess.check_output([str(VENV_PY), '-c',
    'import torch; assert torch.cuda.is_available(); x=torch.ones((16,16),device="cuda",dtype=torch.bfloat16); y=x@x; torch.cuda.synchronize(); print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0), float(y[0,0]))'],
    text=True, env=env)
(RESULT_DIR / 'gpu_import_check.txt').write_text(gpu_check)
print('Environment preflight passed:', gpu_check.strip())

## 4. Reuse or download the pinned BAGEL checkpoint locally

A second run in the same VM reuses `/content/hf_cache_bagel`. Every safetensors header is validated before a server starts.

In [ ]:
manifest_path = RESULT_DIR / 'checkpoint_manifest.json'
subprocess.run([str(VENV_PY), str(REPO / 'experiments/bagel/scripts/download_checkpoint.py'),
                '--cache-dir', str(CACHE_DIR), '--manifest', str(manifest_path)], cwd=REPO, env=env, check=True)
manifest = json.loads(manifest_path.read_text())
print('Validated checkpoint:', manifest['resolved_revision'])

## 5. Run the complete three-arm diagnostic

Run this cell once and leave it running while heartbeats continue. Each arm gets a fresh owned server. On failure or interruption, all evidence collected so far is packaged and the owned process group is stopped. Do not rerun this cell with the same `RUN_ID`.

In [ ]:
sys.path.insert(0, str(REPO / 'experiments/bagel/scripts'))
from experiment_runtime import package_results, run_logged
command = [str(VENV_PY), str(REPO / 'experiments/bagel/scripts/run_batch_shape_diagnostic.py'),
           '--repo-root', str(REPO), '--cache-dir', str(CACHE_DIR),
           '--output-dir', str(DIAGNOSTIC_DIR), '--port', '8027']
run_error = None
try:
    run_logged(command, RESULT_DIR / 'diagnostic_console.log', cwd=REPO, env=env,
               stage='complete Step 7 batch-shape diagnostic', timeout=14400)
except BaseException as exc:
    run_error = exc
finally:
    archive_path = RESULT_DIR.parent / f'{RESULT_DIR.name}_results.zip'
    if archive_path.exists():
        print('Preserving existing local ZIP:', archive_path)
    else:
        receipt = package_results(RESULT_DIR, DRIVE_BACKUP_DIR)
        receipt_path = RESULT_DIR.parent / f'{RESULT_DIR.name}_backup_receipt.json'
        receipt_path.write_text(json.dumps(receipt, indent=2) + '\n')
        print(json.dumps(receipt, indent=2))
if run_error is not None:
    raise run_error
print('Diagnostic completed:', DIAGNOSTIC_DIR)

## 6. Inspect the result and download the verified local ZIP

`batching_general_candidate` means both B=2 arms diverged from B=1, so ordinary B=2 shape is sufficient to produce drift. `mixed_age_specific_candidate` means the homogeneous arm stayed exact while the heterogeneous arm diverged. Neither status silently accepts the numerical difference.

In [ ]:
report = json.loads((DIAGNOSTIC_DIR / 'batch_shape_diagnostic.json').read_text())
compact = {
    'status': report['status'],
    'performance_authorized': report['performance_authorized'],
    'hypothesis_assessment': report['hypothesis_assessment'],
    'trace_contracts': report['trace_contracts'],
    'latent_summary': {arm: [
        {'request_index': row['request_index'], 'seed': row['seed'],
         'first_nonexact_step': row['first_nonexact_step'],
         'final_relative_l2': row['final_relative_l2'],
         'maximum_relative_l2': row['maximum_relative_l2']}
        for row in rows
    ] for arm, rows in report['latent_comparisons'].items()},
    'image_comparisons': report['image_comparisons'],
}
print(json.dumps(compact, indent=2))
archive_path = RESULT_DIR.parent / f'{RESULT_DIR.name}_results.zip'
with zipfile.ZipFile(archive_path) as archive:
    assert archive.testzip() is None
digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
print('Verified ZIP:', archive_path, 'SHA-256:', digest)
from google.colab import files
files.download(str(archive_path))

After the browser download starts, share the ZIP and the printed compact report. Then use **Runtime → Disconnect and delete runtime**. Do not launch a performance sweep from this notebook.